In [4]:
# data processing
import numpy as np
import pandas as pd

# text processing and NLP
import nltk
from textblob import TextBlob

# gensim for topic modeling
import gensim
from gensim import corpora
from gensim.utils import simple_preprocess

# viz
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
from wordcloud import WordCloud, STOPWORDS
import seaborn as sns
import matplotlib.pyplot as plt

# misc
import re
import warnings
warnings.filterwarnings("ignore")

print('Packages successfully imported.')

Packages successfully imported.


In [5]:
df = pd.read_csv('netflix_reviews.csv')

print("Shape of the dataset:", df.shape)
print("Columns in the dataset:", df.columns)

df.head()

Shape of the dataset: (113068, 8)
Columns in the dataset: Index(['reviewId', 'userName', 'content', 'score', 'thumbsUpCount',
       'reviewCreatedVersion', 'at', 'appVersion'],
      dtype='str')


,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion
0,efd00499-5e00-49b5-9f32-bc7177ac5ca6,Mikel Magnusson,Netfix Canada forced my wife into a screen tha...,1,0,8.93.1 build 4 50540,2024-06-14 21:47:49,8.93.1 build 4 50540
1,be0d97e1-7de1-4f07-b493-35a53098b5a4,John McDevitt,I use this app until it asks if I'm still ther...,2,0,8.119.0 build 11 50706,2024-06-14 21:33:01,8.119.0 build 11 50706
2,8970dbcd-d75f-4016-bb93-efa5de3ef9e6,Mayur Savaliya,Boycott Netflix from Bharat,1,1,8.14.0 build 5 40129,2024-06-14 21:31:07,8.14.0 build 5 40129
3,a288bc3c-8a90-42d3-b585-1c8078faa96c,Magdalena Glessing,Little good movies and a lot of wonderful TV s...,5,0,8.118.1 build 10 50703,2024-06-14 21:27:26,8.118.1 build 10 50703
4,c388a806-0795-4812-b04e-5b2cdf327157,Elizabeth Turner,"New to this but, so far smooth sailing.app is ...",5,0,8.118.1 build 10 50703,2024-06-14 21:22:13,8.118.1 build 10 50703


In [6]:
def preprocess_text(text):
    if isinstance(text, float):
        return ""
    text = text.lower()  # lowercase
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.strip()  # remove leading/trailing whitespace
    return text

In [7]:
df['content_c'] = df['content'].apply(preprocess_text)
df.head()

,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,content_c
0,efd00499-5e00-49b5-9f32-bc7177ac5ca6,Mikel Magnusson,Netfix Canada forced my wife into a screen tha...,1,0,8.93.1 build 4 50540,2024-06-14 21:47:49,8.93.1 build 4 50540,netfix canada forced my wife into a screen tha...
1,be0d97e1-7de1-4f07-b493-35a53098b5a4,John McDevitt,I use this app until it asks if I'm still ther...,2,0,8.119.0 build 11 50706,2024-06-14 21:33:01,8.119.0 build 11 50706,i use this app until it asks if im still there...
2,8970dbcd-d75f-4016-bb93-efa5de3ef9e6,Mayur Savaliya,Boycott Netflix from Bharat,1,1,8.14.0 build 5 40129,2024-06-14 21:31:07,8.14.0 build 5 40129,boycott netflix from bharat
3,a288bc3c-8a90-42d3-b585-1c8078faa96c,Magdalena Glessing,Little good movies and a lot of wonderful TV s...,5,0,8.118.1 build 10 50703,2024-06-14 21:27:26,8.118.1 build 10 50703,little good movies and a lot of wonderful tv s...
4,c388a806-0795-4812-b04e-5b2cdf327157,Elizabeth Turner,"New to this but, so far smooth sailing.app is ...",5,0,8.118.1 build 10 50703,2024-06-14 21:22:13,8.118.1 build 10 50703,new to this but so far smooth sailingapp is ea...
